# 18.6 教師與學生的 token 對得上嗎？


教師與學生都輸出兩個機率，看形狀相同就能直接比較嗎？若教師字表是 `[貓,狗]`，學生卻是 `[狗,貓]`，第0欄代表的候選不同。把兩邊第0欄硬對上，會教學生把教師給貓的機率配給狗。這不是學生尚未學好，而是我們把答案名稱弄錯了。我們希望兩邊都把貓的80%對上貓。[18.2](https://birdhackor.github.io/tiny-perceptron-vlm/18.2.html)讓教師與學生用同一張264項的字表，欄數相同；但欄數相同之外，還需要每一欄的候選身份一致，不能只看ID數字。

教師比例 `[0.8,0.2]`，學生比例 `[0.2,0.8]`，看數值似乎差很多；但兩者都對貓給80%、狗給20%，其實語意完全相同。如果只是字表順序交換，可以先把學生的欄位重排成教師的順序，再比較。若兩者切詞的粒度也不同，例如教師用一個「小貓」token、學生用「小」「貓」兩個，就不再是交換欄位能解決的。

圖上先列出兩邊自己的ID對應，下面按「貓、狗」的身份配對。教師ID0的貓要對學生ID1；教師ID1的狗要對學生ID0。因此取學生欄的順序是 `[1,0]`，才得到教師順序下的 `[0.8,0.2]`。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-18-vocab-alignment.svg")))

In [ ]:
import torch

teacher_vocab = ["貓", "狗"]
student_vocab = ["狗", "貓"]
p = torch.tensor([0.8, 0.2])
q = torch.tensor([0.2, 0.8])
order = [student_vocab.index(token) for token in teacher_vocab]
reordered = q[order]
print("重排索引", order)
print("錯配MAE", round((p - q).abs().mean().item(), 4))
print("重排後", reordered.tolist(), "MAE", round((p - reordered).abs().mean().item(), 4))

`index(token)`找出該詞在學生字表的欄號，依教師字表的順序建立`order`，應為 `[1,0]`。`q[order]`沿候選軸按這個順序取欄。MAE是mean absolute error（平均絕對差）：逐欄相減後，`.abs()`取絕對值、`.mean()`求平均、`.item()`取出單一的Python數字。錯配時是`(|0.8-0.2|+|0.2-0.8|)/2=0.6`，重排後每欄相同，因此為0。這個MAE只作配對檢查，不是[18.8](https://birdhackor.github.io/tiny-perceptron-vlm/18.8.html)的蒸餾KL；只要同一欄的意義錯了，後面任何逐欄計算的代價都會被誤導。

此外，雙方要在同一前文、同一個待預測位置比較。教師已經讀到「小貓追」，學生只讀到「小貓」，即使字表相同，分布也在回答不同的問題。EOS與角色標記這些[特殊token](https://birdhackor.github.io/tiny-perceptron-vlm/6.6.html)也要有相同的身份和字表對齊規則；EOS表示文字結束，角色標記辨認說話者，它們不是普通字詞。字表項數一致只是必要條件之一，不是充分保證。

練習只把程式的 `student_vocab` 改成 `["貓", "狗"]`，`q` 也同步改成 `torch.tensor([0.8, 0.2])`。先預測`order`為 `[0,1]`，錯配與重排後的MAE都為0，再執行核對。若只改字表不改q，你就給同一組數字換了意義，不能稱為完成字表對齊。

<details>
<summary>選讀：實驗如何讓兩邊對得上</summary>

本專案的白盒訓練程式固定讓雙方共用ByteTokenizer：雙方用同一套文字單位與特殊token的ID約定，讓各ID與位置更容易一致。byte切詞的方式可回看[6.7](https://birdhackor.github.io/tiny-perceptron-vlm/6.7.html)，本節只需要知道雙方共用轉換規則。使用不同tokenizer的白盒方法，需要另外設計語意與位置的映射，本節不假裝交換欄位就能解決；[黑盒文字蒸餾](https://birdhackor.github.io/tiny-perceptron-vlm/18.3.html)則可先拿文字，再由學生切詞，避開兩份字表的逐欄比較。

屬性教師與寬16、寬32的學生都用相同的264-ID byte字表，角色與EOS的約定也相同。CE+KL那支學生以各位置的`labels != -100`挑出有效目標的分數；教師快取按同一筆題目、同一份標準序列排好，核對有效目標數後才計算，問題和PAD不直接算KL。學教師生成回答的那支，則另外保存教師自由生成的ID，沒有拿它不同的前文去逐行配這份白盒快取。

</details>
